In [1]:
## Load OS module
import os

## Add environment variable for R installation
#os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"
os.environ["PATH"] = r"C:\Users\ArPa3547\AppData\Local\Programs\R\R-4.6.0\bin\x64"

## Import pry2 functions
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import FloatVector, IntVector, globalenv, ListVector, DataFrame, StrVector, BoolVector
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
import rpy2.rinterface as ri

## Initialise R session in background
ri.initr()

## Import other modules
import pandas as pd
import geopandas as gpd
import matplotlib as plt
import xarray as xr
import rioxarray
import numpy as np
from types import SimpleNamespace

## Import healthiar as Python wrapper
healthiar = importr("healthiar")

In [2]:
## Function to convert python lists and pandas dataframes to rpy2 lists and data.frames
def py_to_r(obj):
    if isinstance(obj, (tuple, list, pd.Series)):
        if all(isinstance(item, int) for item in obj):
            return IntVector(obj)
        elif all(isinstance(item, float) for item in obj):
            return FloatVector(obj)
        elif all(isinstance(item, str) for item in obj):
            return StrVector(obj)
        else:
            raise ValueError("Conversion not possible for tuple/list with these element types.")
    elif isinstance(obj, pd.DataFrame):
        with (ro.default_converter + pandas2ri.converter).context():
            return ro.conversion.get_conversion().py2rpy(obj)
    else:
        raise ValueError("Conversion not possible for this object type.")

In [3]:
## Function to convert rpy2 lists and data.frames to python lists and pandas dataframes
def r_to_py(obj):
    """
    Recursive conversion
    """
    if isinstance(obj, DataFrame):
        with localconverter(ro.default_converter + pandas2ri.converter):
            return ro.conversion.rpy2py(obj)
    
    elif isinstance(obj, ListVector):
        return {name: r_to_py(obj.rx2(name)) for name in obj.names}
    
    elif isinstance(obj, (IntVector, FloatVector, StrVector, BoolVector)):
        if len(obj) == 1:
            return obj[0]  # single elements
        else:
            return list(obj)  # multiple elements
    else:
        return obj  # fallback for other types


In [4]:
## Get Python data
canton = ["Zurich", "Basel", "Geneva", "Ticino", "Jura"]
language = ["German","German","French","Italian","French"]
exposure = [11, 11, 10, 8, 7]
burden = [4000, 2500, 3000, 1500, 500]

## Convert input to rpy2 format
r_canton = py_to_r(canton)
r_language = py_to_r(language)
r_exposure = py_to_r(exposure)
r_burden = py_to_r(burden)

## Call healthiar function
results_iteration = healthiar.attribute_health(
    # Names of Swiss cantons
    geo_id_micro = r_canton,
    # Names of languages spoken in the selected Swiss cantons
    geo_id_macro = r_language,
    rr_central = 1.369,
    rr_increment = 10, 
    cutoff_central = 5,
    erf_shape = "log_linear",
    exp_central = r_exposure,
    bhd_central = r_burden
)

## Convert output to Python format
py_results_iteration = r_to_py(results_iteration)
py_results_iteration["health_main"][["geo_id_macro", "impact_rounded", "erf_ci", "exp_ci", "bhd_ci"]].head()

,geo_id_macro,impact_rounded,erf_ci,exp_ci,bhd_ci
1,German,1116.0,central,central,central
2,French,466.0,central,central,central
3,Italian,135.0,central,central,central


In [5]:
results_iteration = healthiar.attribute_health(
    # Names of Swiss cantons
    geo_id_micro = py_to_r(["Zurich", "Basel", "Geneva", "Ticino", "Jura"]),
    # Names of languages spoken in the selected Swiss cantons
    geo_id_macro = py_to_r(["German","German","French","Italian","French"]),
    rr_central = 1.369,
    rr_increment = 10, 
    cutoff_central = 5,
    erf_shape = "log_linear",
    exp_central = py_to_r([11, 11, 10, 8, 7]),
    bhd_central = py_to_r([4000, 2500, 3000, 1500, 500])
)

## Convert output to Python format
py_results_iteration = r_to_py(results_iteration)
py_results_iteration["health_main"][["geo_id_macro", "impact_rounded", "erf_ci", "exp_ci", "bhd_ci"]].head()

,geo_id_macro,impact_rounded,erf_ci,exp_ci,bhd_ci
1,German,1116.0,central,central,central
2,French,466.0,central,central,central
3,Italian,135.0,central,central,central


In [4]:
from scipy.interpolate import CubicSpline

## define ERF with decorator ('@')
@ri.rternalize
def erf_fun(x):
    cs = CubicSpline(
      x = [0, 5, 10, 15, 20, 25, 30, 50, 70, 90, 110],
      y = [1.00, 1.04, 1.08, 1.12, 1.16, 1.20, 1.23, 1.35, 1.45, 1.53, 1.60]
    )
    return float(cs(x)[0])

## pass ERF to attribute_health()
results_pm_copd_mr_brt = healthiar.attribute_health(
  exp_central = 8.85,
  bhd_central = 30747,
  cutoff_central = 0,
  erf_eq_central = erf_fun
)

## convert result to Python format
#py_results_pm_copd_mr_brt = r_to_py(results_pm_copd_mr_brt)